# LoRA vs Full Fine-Tuning — BanglaBERT / XNLI-bn
Corrected pipeline: text normalization + tuned LR + early-stopping-based epoch selection (`best_epoch` logged separately from the epoch ceiling).

Run cells in order. **Restart runtime after Step 3**, then continue from Step 4.

## STEP 1 — Mount Drive, initialize results_log.csv with the corrected schema

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/thesis_bangla_nli"
os.makedirs(PROJECT_DIR, exist_ok=True)

RESULTS_CSV = os.path.join(PROJECT_DIR, "results_log.csv")
LOG_PATH = RESULTS_CSV  # alias — run_experiment() below reads/writes via LOG_PATH

COLUMNS = ["method", "fraction", "seed", "epochs", "learning_rate",
           "best_epoch", "accuracy", "f1_macro", "run_id"]

if not os.path.exists(RESULTS_CSV):
    pd.DataFrame(columns=COLUMNS).to_csv(RESULTS_CSV, index=False)
else:
    existing = pd.read_csv(RESULTS_CSV)
    missing = [c for c in COLUMNS if c not in existing.columns]
    if missing:
        raise ValueError(
            f"results_log.csv exists but is missing columns {missing}. "
            f"Fix the schema before running anything else — do not patch this "
            f"silently or you will lose best_epoch on every future row."
        )

print("Results file:", RESULTS_CSV)
print(pd.read_csv(RESULTS_CSV))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results file: /content/drive/MyDrive/thesis_bangla_nli/results_log.csv
     method  fraction  seed  epochs  best_epoch  learning_rate  accuracy  \
0      lora      0.05     7       8         5.0        0.00020  0.761885   
1   full_ft      0.01    42       3         3.0        0.00002  0.704423   
2      lora      0.01    42       8         5.0        0.00020  0.725506   
3   full_ft      0.01     7       3         2.0        0.00002  0.696982   
4      lora      0.01     7       8         4.0        0.00020  0.692848   
5   full_ft      0.05    42       3         3.0        0.00002  0.758165   
6      lora      0.05    42       8         8.0        0.00020  0.767259   
7   full_ft      0.05     7       3         2.0        0.00002  0.764365   
8   full_ft      0.10    42       3         3.0        0.00002  0.772633   
9      lora      0.10    42       8     

## STEP 2 — Clear old checkpoints + old diagnostics log (clean slate)

In [2]:
import shutil

CHECKPOINT_ROOT = f"{PROJECT_DIR}/checkpoints"
CLEAR_CHECKPOINTS = False  # set True only for a deliberate one-time full wipe

if CLEAR_CHECKPOINTS and os.path.exists(CHECKPOINT_ROOT):
    shutil.rmtree(CHECKPOINT_ROOT)
    print("Checkpoints cleared.")

os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
print(f"CHECKPOINT_ROOT = {CHECKPOINT_ROOT} (cleared this run: {CLEAR_CHECKPOINTS})")

CHECKPOINT_ROOT = /content/drive/MyDrive/thesis_bangla_nli/checkpoints (cleared this run: False)


## STEP 3 — Install deps, restart runtime after this cell

In [3]:
!pip install -q peft --upgrade
!pip install -q torchao --upgrade
!pip install -q git+https://github.com/csebuetnlp/normalizer
!pip install -q evaluate
# ---- RESTART RUNTIME NOW, then continue from Step 4 ----

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00


## STEP 4 — Reload Drive mount, dataset, tokenizer, normalization, metrics

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from normalizer import normalize
import numpy as np
import evaluate

dataset = load_dataset("csebuetnlp/xnli_bn", revision="refs/convert/parquet")
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")

def normalize_and_tokenize(examples):
    s1 = [normalize(t) for t in examples["sentence1"]]
    s2 = [normalize(t) for t in examples["sentence2"]]
    return tokenizer(s1, s2, truncation=True, padding="max_length", max_length=128)

tokenized_ds = dataset.map(normalize_and_tokenize, batched=True)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }
# NOTE: the key must be exactly "f1_macro" — run_experiment() below filters
# trainer.state.log_history on this exact string. Rename it here AND in
# run_experiment() together, or the best-epoch extraction silently breaks.

xnli_bn/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 74.6MB            

xnli_bn/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/242k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/480k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/381449 [00:00<?, ? examples/s]

Map:   0%|          | 0/2419 [00:00<?, ? examples/s]

Map:   0%|          | 0/4895 [00:00<?, ? examples/s]

## STEP 5 — run_experiment() with early stopping + best_epoch logging

In [4]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType

def load_logged_run_ids():
    log_df = pd.read_csv(LOG_PATH)
    return set(log_df["run_id"].tolist())

def write_or_replace_row(log_row):
    """Overwrites the FULL row for a run_id if it already exists — never
    just patches the metric columns, or a rerun can leave a stale
    best_epoch sitting next to fresh accuracy numbers."""
    log_df = pd.read_csv(LOG_PATH)
    log_df = log_df[log_df["run_id"] != log_row["run_id"]]  # drop old row if present
    new_row_df = pd.DataFrame([log_row])
    log_df = new_row_df if log_df.empty else pd.concat([log_df, new_row_df], ignore_index=True)
    log_df.to_csv(LOG_PATH, index=False)

def build_model(method):
    base = AutoModelForSequenceClassification.from_pretrained(
        "csebuetnlp/banglabert", num_labels=3
    )
    if method == "lora":
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            target_modules=["query", "value"],
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
        )
        return get_peft_model(base, lora_config)
    return base  # full fine-tuning

def run_experiment(method, fraction, seed, epochs, learning_rate, force_rerun=False):
    run_id = f"{method}_frac{fraction}_seed{seed}"
    logged_run_ids = load_logged_run_ids()

    if not force_rerun and run_id in logged_run_ids:
        print(f"[SKIP] {run_id} already logged.")
        return

    print(f"[RUN] {run_id} — ceiling={epochs} epochs, lr={learning_rate}")

    n_train = int(len(tokenized_ds["train"]) * fraction)
    train_subset = tokenized_ds["train"].shuffle(seed=seed).select(range(n_train))
    val_subset = tokenized_ds["validation"]

    model = build_model(method)

    training_args = TrainingArguments(
        output_dir=f"{CHECKPOINT_ROOT}/{run_id}",
        num_train_epochs=epochs,          # this is a CEILING, not a target
        learning_rate=learning_rate,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=seed,
        save_total_limit=2,               # don't fill Drive with every checkpoint
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset,
        eval_dataset=val_subset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    import glob
    existing_checkpoints = glob.glob(f"{CHECKPOINT_ROOT}/{run_id}/checkpoint-*")
    resume = bool(existing_checkpoints)
    trainer.train(resume_from_checkpoint=resume)

    eval_logs = [log for log in trainer.state.log_history if "eval_f1_macro" in log]
    if not eval_logs:
        raise RuntimeError(
            f"[{run_id}] No eval_f1_macro entries in log_history. "
            f"compute_metrics key mismatch or evaluation_strategy is wrong — "
            f"fix this before logging a row, don't silently skip it."
        )
    best = max(eval_logs, key=lambda x: x["eval_f1_macro"])

    log_row = {
        "method": method,
        "fraction": fraction,
        "seed": seed,
        "epochs": epochs,                 # ceiling you configured
        "best_epoch": best["epoch"],      # what actually happened
        "learning_rate": learning_rate,
        "accuracy": best.get("eval_accuracy"),
        "f1_macro": best["eval_f1_macro"],
        "run_id": run_id,
    }
    write_or_replace_row(log_row)

    # Save the full epoch-by-epoch curve so a capped run can be diagnosed
    # without retraining — was it basically flat, or still climbing?
    import json
    curve_path = f"{PROJECT_DIR}/curves/{run_id}.json"
    os.makedirs(f"{PROJECT_DIR}/curves", exist_ok=True)
    with open(curve_path, "w") as f:
        json.dump(eval_logs, f, indent=2)

    if best["epoch"] == epochs:
        deltas = [eval_logs[i]["eval_f1_macro"] - eval_logs[i-1]["eval_f1_macro"]
                  for i in range(1, len(eval_logs))]
        last_delta = deltas[-1] if deltas else 0
        tag = "Still climbing meaningfully, ceiling likely too low." if last_delta > 0.003 else "Gain was small — probably near-converged despite hitting the cap."
        print(f"[CEILING HIT] {run_id} — last-epoch F1 gain was {last_delta:+.4f}. {tag}")

    print(f"[LOGGED] {run_id} -> best_epoch={best['epoch']}, f1_macro={best['eval_f1_macro']:.4f}")

In [ ]:
import shutil; shutil.copy(RESULTS_CSV, RESULTS_CSV.replace('.csv', '_BACKUP_pre_wipe.csv'))

In [ ]:
pd.DataFrame(columns=COLUMNS).to_csv(RESULTS_CSV, index=False)

## STEP 6 — LR sanity check (already done — kept here for reference/rerun)

In [ ]:
print(CHECKPOINT_ROOT)

In [ ]:
import json, csv

curves = json.load(open("/content/drive/MyDrive/thesis_bangla_nli/curves/all_curves.json"))
csv_rows = {r["run_id"]: r for r in csv.DictReader(open("/content/drive/MyDrive/thesis_bangla_nli/results_log.csv"))}

for run_id, curve in curves.items():
    expected = int(float(csv_rows[run_id]["epochs"]))
    got = len(curve)  # adjust if your curve is a dict/list with a different top-level shape
    if got != expected:
        print("MISMATCH:", run_id, "expected", expected, "got", got)

In [ ]:
import json
curves = json.load(open("/content/drive/MyDrive/thesis_bangla_nli/curves/all_curves.json"))
c = curves["lora_frac0.05_seed42"]
print(c)  # or however your schema nests it — e.g. c might be {"1": {...}, "2": {...}} or a list

In [ ]:
def epoch_ceiling(method, fraction):
    if method == "lora":
        return 8 if fraction <= 0.10 else 4
    return 3 if fraction <= 0.10 else 2  # full_ft
def lr_for(method):
    return 2e-4 if method == "lora" else 2e-5
run_experiment(
    method="lora",
    fraction=0.05,
    seed=7,
    epochs=epoch_ceiling("lora", 0.05),   # 8
    learning_rate=lr_for("lora"),          # 2e-4
    force_rerun=True,
)

[RUN] lora_frac0.05_seed7 — ceiling=8 epochs, lr=0.0002


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[tra

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.730170,0.705880,0.701943,0.703575
2,0.652808,0.648483,0.748243,0.750311


## STEP 7 — Full matrix loop (24 runs: 6 fractions x 2 seeds x 2 methods)

In [ ]:
FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
SEEDS = [42, 7]

def epoch_ceiling(method, fraction):
    if method == "lora":
        return 8 if fraction <= 0.10 else 4
    return 3 if fraction <= 0.10 else 2  # full_ft

def lr_for(method):
    return 2e-4 if method == "lora" else 2e-5

for fraction in FRACTIONS:
    for seed in SEEDS:
        for method in ["full_ft", "lora"]:
            run_experiment(
                method=method,
                fraction=fraction,
                seed=seed,
                epochs=epoch_ceiling(method, fraction),
                learning_rate=lr_for(method),
                force_rerun=False,   # first corrected pass — flip to False after it completes
            )

## STEP 8 — Resuming after a disconnect

In [ ]:
# Re-run Steps 1, 4, 5 to restore state, then re-run Step 7 with
# force_rerun=False. Already-logged run_ids will be skipped automatically.

In [ ]:
n_val = 2419
pairs = [
    ("lora_frac0.1_seed42", 0.7837949565936337),
    ("lora_frac0.1_seed7",  0.7837949565936337),
    ("full_ft_frac0.25_seed7", 0.7970235634559736),
    ("lora_frac0.25_seed7",    0.7970235634559736),
]
for run_id, acc in pairs:
    correct = acc * n_val
    print(run_id, correct, round(correct))

In [ ]:

import json, glob, os

curves_dir = "/content/drive/MyDrive/thesis_bangla_nli/curves"  # adjust to actual path
merged = {}
for path in glob.glob(f"{curves_dir}/*.json"):
    run_id = os.path.splitext(os.path.basename(path))[0]
    with open(path) as f:
        merged[run_id] = json.load(f)

out_path = f"{curves_dir}/all_curves.json"
with open(out_path, "w") as f:
    json.dump(merged, f)

print(len(merged), "runs merged ->", out_path)


In [ ]:
import glob
glob.glob("/content/drive/MyDrive/thesis_bangla_nli/curves/*lora*frac0.05*seed7*")